# Approximate nearest neighbors (FAISS, HNSW, IVF) — runnable notebook
One focused concept, **5 runnable & visualizable examples** — each cell computes, plots, and asserts. Run-all safe.

In [ ]:
import numpy as np, matplotlib.pyplot as plt, math, itertools
np.random.seed(0)
def cos(a,b):
    a=np.asarray(a,float); b=np.asarray(b,float)
    return float(np.dot(a,b)/(np.linalg.norm(a)*np.linalg.norm(b)))
print('setup ok')

## Search a small candidate set instead of every vector
These examples compare exact nearest-neighbor search with IVF-style clusters, a bad approximation, an HNSW-like greedy walk, and the recall-latency tradeoff.

In [ ]:
# 1) Exact nearest neighbor by Euclidean distance
pts=np.array([[0.,0.],[1.,0.],[0.,1.],[3.,3.],[3.2,3.1],[4.,3.]])
q=np.array([3.1,3.0]); dist=np.linalg.norm(pts-q,axis=1)
plt.figure(figsize=(4,4)); plt.scatter(pts[:,0],pts[:,1]); plt.scatter([q[0]],[q[1]],c='r',marker='x',s=100); plt.title('exact NN is [3,3]')
assert int(np.argmin(dist))==3 and abs(dist[3]-0.1)<1e-9

In [ ]:
# 2) IVF-style coarse partition searches the nearest centroid's list
cent=np.array([[1/3,1/3],[3.4,3.0333333333]]); cd=np.linalg.norm(cent-q,axis=1); lists=[[0,1,2],[3,4,5]]; cand=lists[int(np.argmin(cd))]
plt.figure(figsize=(4,4)); plt.scatter(pts[:,0],pts[:,1],c=['tab:blue']*3+['tab:orange']*3); plt.scatter(cent[:,0],cent[:,1],c='k',marker='s'); plt.scatter([q[0]],[q[1]],c='r',marker='x'); plt.title('search only the nearest coarse list')
assert cand==[3,4,5] and abs(cd[1]-0.3018467368)<1e-6

In [ ]:
# 3) Approximation can be measured by recall@1
ann_idx=cand[int(np.argmin(dist[cand]))]; bad_cand=[5]; bad_idx=bad_cand[int(np.argmin(dist[bad_cand]))]
plt.figure(figsize=(6,3)); plt.bar(['good candidate set','bad candidate set'],[ann_idx==np.argmin(dist),bad_idx==np.argmin(dist)]); plt.ylim(0,1.1); plt.title('recall@1 exposes misses')
assert ann_idx==3 and bad_idx!=3

In [ ]:
# 4) HNSW-like greedy walk follows neighbor links downhill
neighbors={0:[1,2],1:[0,3],2:[0,3],3:[1,2,4,5],4:[3,5],5:[3,4]}
path=[0]; cur=0
while True:
    opts=neighbors[cur]+[cur]; nxt=min(opts,key=lambda i: dist[i])
    if nxt==cur: break
    path.append(nxt); cur=nxt
plt.figure(figsize=(4,4)); plt.scatter(pts[:,0],pts[:,1]); plt.plot(pts[path,0],pts[path,1],'-o',c='r'); plt.scatter([q[0]],[q[1]],marker='x',c='k'); plt.title('greedy graph walk toward the query')
assert path[-1]==3 and len(path)<len(pts)

In [ ]:
# 5) More probes cost more comparisons but improve recall chance
probes=np.array([1,2]); comps=np.array([3,6]); recall=np.array([1,1])
plt.figure(figsize=(6,3)); plt.bar(probes-.1,comps,width=.2,label='comparisons'); plt.bar(probes+.1,recall,width=.2,label='recall@1'); plt.xticks(probes); plt.legend(); plt.title('probe more lists: more work, safer recall')
assert comps[0]<comps[1] and recall[0]==1